from last ADS-B point see how far range has to be to get 99.9% of airports within the candiates. I can distance do this based on altitude. Maybe use average altitude of the airports nearby

In [1]:
5+5

10

In [2]:
from datetime import date
from evaluation.flights.evaluation import main
import polars as pl
test_date = date(2026,3,1)
outputs = main(test_src="model", gold_src="sfdps",pia_or_american_ladd_only=True, test_dates=test_date)
df, stats = outputs[0]

Loading airports from /Volumes/T2-SSD/planequery/data/interim/airports.csv


ValueError: Unknown flight source: model

In [ ]:
COLUMNS = [
    "icao",
    "first_message_time",
    "first_lat",
    "first_lon",
    "first_baro_altitude_ft",
    "last_message_time",
    "last_lat",
    "last_lon",
    "last_baro_altitude_ft",
    "takeoff_time_df1",
    "landing_time_df1",
    "takeoff_airport_ident",
    "landing_airport_ident",
    "takeoff_airport_ident_df1",
    "landing_airport_ident_df1",
    "same_takeoff_airport_ident",
    "same_landing_airport_ident",
    "same_airport_ident",
    'match_status'
]
df = df.select(COLUMNS)

In [ ]:
import math

from airports.airportLookup import AirportLookup
from utils import haversine

AIRPORT_RADIUS_KM = 50.0
airport_lookup = AirportLookup()

def airport_idents_within_radius(row, radius_km=AIRPORT_RADIUS_KM):
    lat = row["lat"]
    lon = row["lon"]
    if lat is None or lon is None or math.isnan(lat) or math.isnan(lon):
        return []

    lat = float(lat)
    lon = float(lon)
    airports = airport_lookup.getAirportsWithinRadius(lat, lon, radius_km=radius_km)
    ident_distances = [
        (airport.ident, haversine(lat, lon, airport.lat, airport.lon))
        for airport in airports
    ]
    return [
        ident
        for ident, distance_km in sorted(ident_distances, key=lambda item: item[1])
        if distance_km <= radius_km
    ]

def airport_ident_in_candidates(row):
    ident = row["ident"]
    candidate_idents = row["candidate_idents"]
    if ident is None or candidate_idents is None:
        return False
    return ident in set(candidate_idents)

df_airport_candidates = (
    df.with_columns(
        pl.struct([
            pl.col("first_lat").alias("lat"),
            pl.col("first_lon").alias("lon"),
        ])
        .map_elements(airport_idents_within_radius, return_dtype=pl.List(pl.Utf8))
        .alias("takeoff_airports_set"),
        pl.struct([
            pl.col("last_lat").alias("lat"),
            pl.col("last_lon").alias("lon"),
        ])
        .map_elements(airport_idents_within_radius, return_dtype=pl.List(pl.Utf8))
        .alias("landing_airports_set"),
    )
    .with_columns(
        pl.col("takeoff_airports_set")
        .list.len()
        .alias("takeoff_airports_set_count"),
        pl.col("landing_airports_set")
        .list.len()
        .alias("landing_airports_set_count"),
        pl.struct([
            pl.col("takeoff_airport_ident_df1").alias("ident"),
            pl.col("takeoff_airports_set").alias("candidate_idents"),
        ])
        .map_elements(airport_ident_in_candidates, return_dtype=pl.Boolean)
        .alias("takeoff_airport_ident_df1_in_takeoff_airports_set"),
        pl.struct([
            pl.col("landing_airport_ident_df1").alias("ident"),
            pl.col("landing_airports_set").alias("candidate_idents"),
        ])
        .map_elements(airport_ident_in_candidates, return_dtype=pl.Boolean)
        .alias("landing_airport_ident_df1_in_landing_airports_set"),
    )
)

df_airport_candidates.select(
    [
        "icao",
        "first_lat",
        "first_lon",
        "takeoff_airport_ident_df1",
        "takeoff_airport_ident_df1_in_takeoff_airports_set",
        "takeoff_airports_set_count",
        "takeoff_airports_set",
        "last_lat",
        "last_lon",
        "landing_airport_ident_df1",
        "landing_airport_ident_df1_in_landing_airports_set",
        "landing_airports_set_count",
        "landing_airports_set",
    ]
)

icao,first_lat,first_lon,takeoff_airport_ident_df1,takeoff_airport_ident_df1_in_takeoff_airports_set,takeoff_airports_set_count,takeoff_airports_set,last_lat,last_lon,landing_airport_ident_df1,landing_airport_ident_df1_in_landing_airports_set,landing_airports_set_count,landing_airports_set
str,f64,f64,str,bool,u32,list[str],f64,f64,str,bool,u32,list[str]
"""a0001a""",33.629313,-111.908183,"""KSDL""",true,15,"[""KSDL"", ""US-1815"", … ""KGYR""]",27.1772,-80.220574,"""KSUA""",true,17,"[""KSUA"", ""64FA"", … ""FD39""]"
"""a00028""",27.179152,-80.222432,"""KSUA""",true,17,"[""KSUA"", ""64FA"", … ""FD39""]",41.057863,-73.699722,"""KHPN""",true,8,"[""KHPN"", ""KLGA"", … ""KBDR""]"
"""a00055""",30.509216,-81.656045,"""KJAX""",true,17,"[""KJAX"", ""FL84"", … ""FL11""]",33.784542,-84.490986,"""KFTY""",true,44,"[""KFTY"", ""KMGE"", … ""17GA""]"
"""a000cc""",28.547653,-81.34243,"""KORL""",true,23,"[""KORL"", ""KMCO"", … ""FA42""]",27.897217,-82.686215,"""KPIE""",true,15,"[""KPIE"", ""KCLW"", … ""23FL""]"
"""a000cc""",27.918549,-82.689791,"""KPIE""",true,15,"[""KPIE"", ""KCLW"", … ""FL78""]",28.54771,-81.330387,"""KORL""",true,23,"[""KORL"", ""KMCO"", … ""KX21""]"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""adf522""",34.215729,-118.484788,"""KVNY""",true,10,"[""KVNY"", ""KWHP"", … ""KTOA""]",32.411087,-111.21861,"""KAVQ""",true,20,"[""KAVQ"", ""US-1654"", … ""US-2562""]"
"""adf528""",33.619235,-116.156445,"""KTRM""",true,10,"[""KTRM"", ""KUDD"", … ""58CL""]",34.933591,-82.189865,"""KGSP""",true,36,"[""KGSP"", ""US-7610"", … ""4SC4""]"
"""adf528""",34.88855,-82.22409,"""KGSP""",true,33,"[""KGSP"", ""SC72"", … ""KLUX""]",32.921836,-80.054907,"""KCHS""",true,14,"[""KCHS"", ""SC98"", … ""US-7189""]"


In [36]:
df_airport_candidates.filter((pl.col("match_status") == "both") & (pl.col("same_landing_airport_ident") == False) & (pl.col("landing_airport_ident_df1_in_landing_airports_set") == False))

icao,first_message_time,first_lat,first_lon,first_baro_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,takeoff_time_df1,landing_time_df1,takeoff_airport_ident,landing_airport_ident,takeoff_airport_ident_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident,match_status,takeoff_airports_set,landing_airports_set,takeoff_airports_set_count,landing_airports_set_count,takeoff_airport_ident_df1_in_takeoff_airports_set,landing_airport_ident_df1_in_landing_airports_set
str,datetime[ms],f64,f64,i64,datetime[ms],f64,f64,i64,datetime[ms],datetime[ms],str,str,str,str,bool,bool,bool,str,list[str],list[str],u32,u32,bool,bool
"""a02988""",2026-03-01 22:54:45.500,32.843814,-96.854684,550,2026-03-01 23:33:56.670,32.648438,-94.759003,5000,2026-03-01 22:55:00,2026-03-01 23:57:00,"""KDAL""","""2XA7""","""KDAL""","""KDTN""",true,false,false,"""both""","[""KDAL"", ""KADS"", … ""8TS1""]","[""2XA7"", ""3TS0"", … ""US-6792""]",47,21,true,false
"""a03b35""",2026-03-01 15:26:04.200,32.817431,-97.353357,null,2026-03-01 16:58:47.350,33.494431,-100.917938,15775,2026-03-01 16:28:00,2026-03-01 17:11:00,"""KFTW""","""83XA""","""KFTW""","""KLBB""",true,false,false,"""both""","[""KFTW"", ""KNFW"", … ""4TX8""]","[""83XA"", ""TA16"", … ""TS21""]",76,11,true,false
"""a2cff3""",2026-03-01 14:03:24.210,26.64061,-81.809019,2450,2026-03-01 19:19:34.470,44.505798,-88.08269,925,2026-03-01 14:02:00,2026-03-01 18:30:00,"""KRSW""","""KGRB""","""KFMY""","""KTVC""",false,false,false,"""both""","[""47FD"", ""KFMY"", … ""FA54""]","[""KGRB"", ""US-7510"", … ""11Y""]",31,26,true,false
"""a49fd2""",2026-03-01 19:47:25.840,40.230316,-111.728476,null,2026-03-01 20:34:44.760,40.110618,-106.653401,17475,2026-03-01 19:51:00,2026-03-01 20:53:00,"""KPVU""","""K20V""","""KPVU""","""KEGE""",true,false,false,"""both""","[""KPVU"", ""KSPK"", … ""KU42""]","[""K20V"", ""5CO8"", … ""KSBS""]",11,5,true,false
"""a8050c""",2026-03-01 19:33:49.680,33.973823,-83.971996,1175,2026-03-01 22:05:54.070,33.315857,-83.673523,3800,2026-03-01 19:34:00,2026-03-01 22:40:00,"""KLZU""","""GA06""","""KLZU""","""KLZU""",true,false,false,"""both""","[""KLZU"", ""2GA9"", … ""GA83""]","[""GA06"", ""US-7849"", … ""GE35""]",29,28,true,false
"""aba71a""",2026-03-01 17:53:40.640,30.03704,-93.227399,3250,2026-03-01 18:54:15.940,28.03125,-97.224039,10450,2026-03-01 17:51:00,2026-03-01 19:26:00,"""KLCH""","""KTFP""","""KLCH""","""KALI""",true,false,false,"""both""","[""7LA9"", ""KLCH"", … ""7LA3""]","[""KTFP"", ""TE87"", … ""TE62""]",24,17,true,false
"""aba882""",2026-03-01 21:29:30,29.788498,-95.898581,700,2026-03-01 21:59:19.750,29.973682,-93.928601,8825,2026-03-01 21:29:00,2026-03-01 22:12:00,"""KTME""","""KBPT""","""KTME""","""KUXL""",true,false,false,"""both""","[""KTME"", ""27XS"", … ""00XA""]","[""KBPT"", ""XS33"", … ""US-6117""]",47,13,true,false
"""ac97c2""",2026-03-01 15:32:43.130,39.080154,-77.558222,300,2026-03-01 15:50:22.880,39.421097,-78.53517,5175,2026-03-01 15:32:00,2026-03-01 16:49:00,"""KJYO""","""WV67""","""KJYO""","""KJYO""",true,false,false,"""both""","[""KJYO"", ""3VA5"", … ""15VA""]","[""WV67"", ""38WV"", … ""61VA""]",27,20,true,false
